In [1]:
import sys
import os
import uuid
import pandas as pd
from pymongo import MongoClient

# إضافة المسار الرئيسي
sys.path.append(os.path.abspath('.'))

from config.settings import MONGO_URI, DB_NAME, COLLECTION_RAW, COLLECTION_VALIDATED, COLLECTION_QUARANTINE
from src.mongo_setup import init_mongo

# تهيئة الفهارس وقواعد البيانات
init_mongo()

# الاتصال بالـ DB للمعاينة
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
print("✅ MongoDB Connected Successfully")

MongoDB collections and indexes initialized.
✅ MongoDB Connected Successfully


In [4]:
from src.create_small_sample import create_sample

input_file = "E:\\level 4th\\term1\\big Data\\midterm-data-pipeline\\data\\orders_huge_mixed_quality.csv"  # استبدلي باسم ملفك
sample_file = "E:\\level 4th\\term1\\big Data\\midterm-data-pipeline\\data\\sample_orders.csv"

# استخراج 5000 سطر
create_sample(input_file, sample_file, num_rows=5000)

# معاينة أول 5 أسطر من العينة
df_sample_preview = pd.read_csv(sample_file, nrows=5)
display(df_sample_preview)

Sample generated successfully: E:\level 4th\term1\big Data\midterm-data-pipeline\data\sample_orders.csv with 5000 rows.


,order_id,order_date,status,customer_id,customer_name,customer_phone,customer_email,city,district,delivery_type,delivery_cost,payment_method,payment_status,payment_amount,currency,total_amount,items_json
0,طلب-100000,2025-02-24T21:29:00,مؤكد,عميل-0,محمد علي,702390941,user141764@example.com,تعز,شعوب,سريع,5000.0,محفظة إلكترونية,تم الدفع,769000.0,YER,769000.0,"[{""sku"":""SKU-1010"",""name"":""هاتف سامسونج A54"",""..."
1,طلب-100001,2025-01-12T00:13:00,قيد الانتظار,عميل-1,علي حسين,714876334,user392083@example.com,لحج,القاهرة,عادي,2000.0,محفظة إلكترونية,بانتظار الدفع,546500.0,YER,546500.0,"[{""sku"":""SKU-1010"",""name"":""هاتف سامسونج A54"",""..."
2,طلب-100002,2025-02-26T20:26:00,مؤكد,عميل-2,أروى محمد,717011292,user697610@example.com,حجة,جبلة,عادي,2000.0,محفظة إلكترونية,تم الدفع,32000.0,YER,32000.0,"[{""sku"":""SKU-1009"",""name"":""شاحن سريع"",""qty"":3,..."
3,طلب-100003,2025-02-22T03:22:00,مرتجع,عميل-3,ريم عادل,739988747,user309803@example.com,عدن,القاهرة,عادي,2000.0,محفظة إلكترونية,تم الدفع,706000.0,YER,٧٠٦٠٠٠٫٠,"[{""sku"":""SKU-1010"",""name"":""هاتف سامسونج A54"",""..."
4,طلب-100004,2025-03-09T19:18:00,قيد الشحن,عميل-4,محمد علي,704925363,user178511@example.com,المكلا,جبلة,سريع,5000.0,بطاقة,تم الدفع,374000.0,YER,374000.0,"[{""sku"":""SKU-1010"",""name"":""هاتف سامسونج A54"",""..."


In [5]:
from src.batch_loader import run_batch_loader

# توليد معرف فريد لعملية التشغيل
current_run_id = str(uuid.uuid4())
print(f"Run ID: {current_run_id}")

# تشغيل التحميل
batch_metrics = run_batch_loader(sample_file, current_run_id)
print(batch_metrics)

# معاينة البيانات الخام التي دخلت في MongoDB
raw_sample = list(db[COLLECTION_RAW].find({"run_id": current_run_id}).limit(3))
for doc in raw_sample:
    print(f"\n--- Row #{doc['number_row_source']} ---")
    print("Metadata:", {k: doc[k] for k in ["run_id", "engine_used", "at_ingested"]})
    print("Record Raw:", doc['record_raw'])

Run ID: 2055e811-dc07-4e6b-a899-521a2d295cff
Starting Python Batch Loading for: E:\level 4th\term1\big Data\midterm-data-pipeline\data\sample_orders.csv
  -> Batch #1 inserted: 1000 records | Time: 0.099s | Rate: 10144.5 rec/s
  -> Batch #2 inserted: 1000 records | Time: 0.011s | Rate: 91558.7 rec/s
  -> Batch #3 inserted: 1000 records | Time: 0.009s | Rate: 112216.2 rec/s
  -> Batch #4 inserted: 1000 records | Time: 0.010s | Rate: 100222.3 rec/s
  -> Batch #5 inserted: 1000 records | Time: 0.009s | Rate: 110723.2 rec/s
Batch loading finished: 5000 records in 0.19s | Avg Throughput: 25950.8 rec/s

{'engine': 'python_batch', 'loaded_raw': 5000, 'seconds_elapsed': 0.1926720142364502, 'throughput': 25950.836813611757, 'batch_size': 1000}

--- Row #1 ---
Metadata: {'run_id': '2055e811-dc07-4e6b-a899-521a2d295cff', 'engine_used': 'python_batch', 'at_ingested': '2026-08-18T14:16:53.903779'}
Record Raw: {'\ufefforder_id': 'طلب-100000', 'order_date': '2025-02-24T21:29:00', 'status': 'مؤكد', 'c

In [6]:
from src.quality_rules import validate_and_clean_record

# تجربة دالة التنظيف على السجلات الخام المسترجعة من الـ Raw Layer
raw_records_cursor = db[COLLECTION_RAW].find({"run_id": current_run_id}).limit(10)

valid_examples = []
corrected_examples = []
quarantine_examples = []

for item in raw_records_cursor:
    res = validate_and_clean_record(item["record_raw"])
    if res["status"] == "valid" and len(valid_examples) < 2:
        valid_examples.append(res["data"])
    elif res["status"] == "corrected" and len(corrected_examples) < 2:
        corrected_examples.append(res["data"])
    elif res["status"] == "quarantine" and len(quarantine_examples) < 2:
        quarantine_examples.append(res["data"])

print("--- مثال على سجل سليم (Valid) ---")
if valid_examples:
    print(valid_examples[0])

print("\n--- مثال على سجل مصحح مع الـ Audit Trail (Corrected) ---")
if corrected_examples:
    import json
    print(json.dumps(corrected_examples[0], ensure_ascii=False, indent=2))

print("\n--- مثال على سجل معزول (Quarantine) ---")
if quarantine_examples:
    print(quarantine_examples[0])

--- مثال على سجل سليم (Valid) ---

--- مثال على سجل مصحح مع الـ Audit Trail (Corrected) ---

--- مثال على سجل معزول (Quarantine) ---
{'order_id': None, 'error_codes': ['ID_ORDER_MISSING', 'DATE_IMPOSSIBLE_INVALID'], 'error_details': 'Failed validations: ID_ORDER_MISSING, DATE_IMPOSSIBLE_INVALID', 'raw_record': {'\ufefforder_id': 'طلب-100000', 'order_date': '2025-02-24T21:29:00', 'status': 'مؤكد', 'customer_id': 'عميل-0', 'customer_name': 'محمد علي', 'customer_phone': '702390941', 'customer_email': 'user141764@example.com', 'city': 'تعز', 'district': 'شعوب', 'delivery_type': 'سريع', 'delivery_cost': '5000.0', 'payment_method': 'محفظة إلكترونية', 'payment_status': 'تم الدفع', 'payment_amount': '769000.0', 'currency': 'YER', 'total_amount': '769000.0', 'items_json': '[{"sku":"SKU-1010","name":"هاتف سامسونج A54","qty":-2,"unit_price":183000.0,"total":549000.0},{"sku":"SKU-1010","name":"هاتف سامسونج A54","qty":1,"unit_price":215000.0,"total":215000.0}]'}}


In [7]:
from src.elt_pipeline import process_elt_transformation

# تشغيل المعالجة على نفس الـ run_id الذي حملناه في الخطوة السابقة
elt_results = process_elt_transformation(current_run_id)

print("\n--- ملخص النتائج والمقاييس ---")
import pprint
pprint.pprint(elt_results)

# التأكد من عدد السجلات في كل Collection
print("\n--- إحصائيات MongoDB الإجمالية ---")
print("Raw Total:", db[COLLECTION_RAW].count_documents({}))
print("Validated Total:", db[COLLECTION_VALIDATED].count_documents({}))
print("Quarantine Total:", db[COLLECTION_QUARANTINE].count_documents({}))

Starting ELT Transformation for run_id: 2055e811-dc07-4e6b-a899-521a2d295cff ...
Consistency Check: PASSED (OK)
  Raw Processed: 5000
  Valid: 0 | Corrected: 0 | Quarantine: 5000
  Upsert Metrics -> Inserted: 0, Updated: 0, Unchanged: 0

--- ملخص النتائج والمقاييس ---
{'consistency_check': True,
 'count_corrected': 0,
 'count_inserted': 0,
 'count_quarantine': 5000,
 'count_unchanged': 0,
 'count_updated': 0,
 'count_valid': 0,
 'counts_case_error': {'DATE_IMPOSSIBLE_INVALID': 5000,
                       'ID_CUSTOMER_MISSING': 70,
                       'ID_ORDER_MISSING': 5000,
                       'ITEMS_EMPTY': 39,
                       'JSON_ITEMS_CORRUPTED': 68},
 'processed_raw': 5000,
 'run_id': '2055e811-dc07-4e6b-a899-521a2d295cff',
 'transformation_seconds': 0.3073568344116211}

--- إحصائيات MongoDB الإجمالية ---
Raw Total: 5000
Validated Total: 0
Quarantine Total: 5000


In [14]:
import re
import json
from datetime import datetime
from pymongo import MongoClient, UpdateOne

# 1. دوال التنظيف المحدثة لمعالجة كافة الحالات
def normalize_arabic_digits(val):
    if not val:
        return val
    arabic_digits = "٠١٢٣٤٥٦٧٨٩٫"
    latin_digits = "0123456789."
    trans = str.maketrans(arabic_digits, latin_digits)
    return str(val).translate(trans)

def clean_thousand_separators(val):
    if not val:
        return val
    val = normalize_arabic_digits(val)
    return re.sub(r'(?<=\d),(?=\d)', '', str(val).strip())

def standardize_currency(val):
    if not val:
        return "YER"
    val = str(val).strip()
    if val in ["لاير", "لاير يمني", "ريال", "ريال يمني", "YER", "YER "]:
        return "YER"
    return val

WORD_TO_NUM = {
    "ألف": 1000, "الف": 1000, "ألفان": 2000, "الفان": 2000, 
    "ألفين": 2000, "الفين": 2000, "ثلاثة آلاف": 3000, "ثلاثة الاف": 3000,
    "أربعة آلاف": 4000, "اربعة الاف": 4000, "خمسة آلاف": 5000, "خمسة الاف": 5000
}
def parse_price_words(val):
    if not val:
        return val
    val_clean = str(val).strip()
    return str(WORD_TO_NUM.get(val_clean, val))

def clean_phone(val):
    if not val:
        return val
    val = normalize_arabic_digits(val)
    return re.sub(r'[\s\-\(\)\+]', '', str(val))

def clean_email(val):
    if not val:
        return val
    val = str(val).strip()
    val = re.sub(r'@+', '@', val)
    return re.sub(r'\.+', '.', val)

def standardize_date(val):
    if not val:
        return None
    val = normalize_arabic_digits(val).strip()
    val = val.replace("T", " ")
    val = re.sub(r'\s*/\s*', '/', val)
    val = re.sub(r'\s*-\s*', '-', val)
    
    formats = [
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d %H:%M:%S", "%Y/%m/%d",
        "%d-%m-%Y %H:%M:%S", "%d-%m-%Y", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y"
    ]
    for fmt in formats:
        try:
            parsed = datetime.strptime(val, fmt)
            if 2000 <= parsed.year <= 2030:
                return parsed.strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def validate_and_clean_record(raw_dict):
    clean_raw = {}
    for k, v in raw_dict.items():
        if k:
            clean_key = k.replace('\ufeff', '').strip()
            clean_raw[clean_key] = v

    corrections = []
    quarantine_reasons = []

    # فحص المفاتيح الأساسية
    order_id = str(clean_raw.get("order_id", "") or "").strip()
    if not order_id or order_id.lower() in ["null", "none", "nan", ""]:
        quarantine_reasons.append("ID_ORDER_MISSING")

    customer_id = str(clean_raw.get("customer_id", "") or "").strip()
    if not customer_id or customer_id.lower() in ["null", "none", "nan", ""]:
        quarantine_reasons.append("ID_CUSTOMER_MISSING")

    # التاريخ
    raw_date = str(clean_raw.get("order_date", "") or "").strip()
    norm_date = standardize_date(raw_date)
    if not norm_date:
        quarantine_reasons.append("DATE_IMPOSSIBLE_INVALID")
    elif norm_date != raw_date:
        corrections.append({
            "field": "order_date",
            "original_value": raw_date,
            "corrected_value": norm_date,
            "rule_code": "DATE_STANDARDIZED"
        })

    # الـ JSON
    items_raw = str(clean_raw.get("items_json", "") or "").strip()
    if not items_raw or items_raw in ["[]", "{}"]:
        quarantine_reasons.append("ITEMS_EMPTY")
    else:
        try:
            json.loads(items_raw)
        except Exception:
            try:
                fixed_json = items_raw.replace("'", '"')
                json.loads(fixed_json)
                corrections.append({
                    "field": "items_json",
                    "original_value": items_raw,
                    "corrected_value": fixed_json,
                    "rule_code": "JSON_QUOTES_FIXED"
                })
                clean_raw["items_json"] = fixed_json
            except Exception:
                quarantine_reasons.append("JSON_ITEMS_CORRUPTED")

    if quarantine_reasons:
        return {
            "status": "quarantine",
            "data": {
                "order_id": order_id if order_id else None,
                "error_codes": quarantine_reasons,
                "error_details": f"Failed validations: {', '.join(quarantine_reasons)}",
                "raw_record": clean_raw
            }
        }

    clean_data = dict(clean_raw)
    clean_data["order_id"] = order_id
    clean_data["customer_id"] = customer_id
    clean_data["order_date"] = norm_date

    # البريد
    raw_email = str(clean_raw.get("customer_email", "") or "")
    c_email = clean_email(raw_email)
    if c_email != raw_email:
        corrections.append({
            "field": "customer_email",
            "original_value": raw_email,
            "corrected_value": c_email,
            "rule_code": "EMAIL_REPEATED_SYMBOLS"
        })
        clean_data["customer_email"] = c_email

    # الهاتف
    raw_phone = str(clean_raw.get("customer_phone", "") or "")
    c_phone = clean_phone(raw_phone)
    if c_phone != raw_phone:
        corrections.append({
            "field": "customer_phone",
            "original_value": raw_phone,
            "corrected_value": c_phone,
            "rule_code": "PHONE_NORMALIZED"
        })
        clean_data["customer_phone"] = c_phone

    # العملة
    raw_curr = str(clean_raw.get("currency", "") or "")
    c_curr = standardize_currency(raw_curr)
    if c_curr != raw_curr:
        corrections.append({
            "field": "currency",
            "original_value": raw_curr,
            "corrected_value": c_curr,
            "rule_code": "CURRENCY_STANDARDIZED"
        })
        clean_data["currency"] = c_curr

    # المبالغ المالية
    for field in ["delivery_cost", "payment_amount", "total_amount"]:
        orig = str(clean_raw.get(field, "") or "")
        parsed_words = parse_price_words(orig)
        cleaned_num = clean_thousand_separators(parsed_words)
        if cleaned_num != orig:
            corrections.append({
                "field": field,
                "original_value": orig,
                "corrected_value": cleaned_num,
                "rule_code": "NUMBER_NORMALIZED"
            })
            clean_data[field] = cleaned_num

    quality_status = "corrected" if len(corrections) > 0 else "valid"
    clean_data["quality_status"] = quality_status
    clean_data["corrections"] = corrections

    return {
        "status": quality_status,
        "data": clean_data
    }

# 2. تنفيذ المعالجة والـ Upsert
raw_cursor = db[COLLECTION_RAW].find({"run_id": new_run_id})

valid_bulk_ops = []
quarantine_docs = []
count_valid = 0
count_corrected = 0
count_quarantine = 0
error_cases_count = {}

for doc in raw_cursor:
    res = validate_and_clean_record(doc.get("record_raw", {}))
    status = res["status"]
    data = res["data"]
    
    if status in ["valid", "corrected"]:
        if status == "valid":
            count_valid += 1
        else:
            count_corrected += 1
        valid_bulk_ops.append(UpdateOne({"order_id": data["order_id"]}, {"$set": data}, upsert=True))
    else:
        count_quarantine += 1
        data["run_id"] = new_run_id
        quarantine_docs.append(data)
        for err in data.get("error_codes", []):
            error_cases_count[err] = error_cases_count.get(err, 0) + 1

# كتابة الـ Upsert في MongoDB
inserted = 0
updated = 0
if valid_bulk_ops:
    bulk_res = db[COLLECTION_VALIDATED].bulk_write(valid_bulk_ops, ordered=False)
    inserted = bulk_res.upserted_count
    updated = bulk_res.modified_count

if quarantine_docs:
    db[COLLECTION_QUARANTINE].insert_many(quarantine_docs, ordered=False)

print("🎯 النتيجة بعد التحديث المباشر:")
print(f"  Valid: {count_valid}")
print(f"  Corrected: {count_corrected}")
print(f"  Quarantine: {count_quarantine}")
print(f"  Upsert Inserted: {inserted}, Updated: {updated}")
print(f"  Total Validated in DB: {db[COLLECTION_VALIDATED].count_documents({})}")

🎯 النتيجة بعد التحديث المباشر:
  Valid: 0
  Corrected: 4776
  Quarantine: 224
  Upsert Inserted: 4737, Updated: 39
  Total Validated in DB: 4737
